In [1]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
 
 
class VectorizedStochasticGridWorld:
    """Builds the MDP: the transition tensor P and reward tensor R."""
 
    def __init__(self, height=10, width=10, slip_prob=0.15, step_cost=-0.1):
        self.height = height
        self.width = width
        self.num_states = height * width
        self.num_actions = 4  # 0: Up, 1: Down, 2: Left, 3: Right
 
        self.goal_state = self.num_states - 1          # tile 99
        self.trap_state = (height // 2) * width + (width // 2)  # tile 55
 
        self.slip_prob = slip_prob      # total probability of "slipping"
        self.step_cost = step_cost      # cost of a normal step
 
        self.P = np.zeros((self.num_states, self.num_actions, self.num_states))
        self.R = np.zeros((self.num_states, self.num_actions))
 
        self.build_world()
 
    def build_world(self):
        intended_prob = 1.0 - self.slip_prob        # e.g. 0.85
        each_slip_prob = self.slip_prob / 3.0        # e.g. 0.05
 
        for s in range(self.num_states):
            # Terminal states: goal and trap are "absorbing" -> robot stays forever
            if s == self.goal_state or s == self.trap_state:
                for a in range(self.num_actions):
                    self.P[s, a, s] = 1.0
                    self.R[s, a] = 0.0   # no more reward once episode has ended
                continue
 
            r, c = s // self.width, s % self.width
 
            # Where would the robot land for each of the 4 possible moves?
            # max()/min() clip against the walls (bumping into a wall = stay put)
            neighbors = {
                0: (max(r - 1, 0), c),                      # Up
                1: (min(r + 1, self.height - 1), c),        # Down
                2: (r, max(c - 1, 0)),                      # Left
                3: (r, min(c + 1, self.width - 1)),         # Right
            }
            neighbor_state = {
                a: nr * self.width + nc for a, (nr, nc) in neighbors.items()
            }
 
            for a in range(self.num_actions):
                intended_s = neighbor_state[a]
                self.P[s, a, intended_s] += intended_prob
 
                for slip_a in range(self.num_actions):
                    if slip_a != a:
                        slip_s = neighbor_state[slip_a]
                        self.P[s, a, slip_s] += each_slip_prob
 
                # Expected reward = sum over possible next states of
                # probability * reward-for-landing-there
                for s_next in range(self.num_states):
                    prob = self.P[s, a, s_next]
                    if prob > 0:
                        if s_next == self.goal_state:
                            self.R[s, a] += prob * 10.0
                        elif s_next == self.trap_state:
                            self.R[s, a] += prob * -10.0
                        else:
                            self.R[s, a] += prob * self.step_cost
 
 
class DynamicProgrammingSolver:
    """Runs Value Iteration and Policy Iteration on a given environment."""
 
    def __init__(self, env, gamma=0.99, theta=1e-8, max_iters=1000):
        self.env = env
        self.gamma = gamma
        self.theta = theta
        self.max_iters = max_iters
 
    def value_iteration(self):
        """
        Bellman optimality update, vectorized with np.tensordot:
            Q(s,a) = R(s,a) + gamma * sum_s' P(s,a,s') * V(s')
            V(s)   = max_a Q(s,a)
        Repeat until V stops changing (delta < theta).
        """
        env = self.env
        V = np.zeros(env.num_states)
        history = []  # track how much V changes each iteration (for curiosity/plots)
 
        for i in range(self.max_iters):
            # np.tensordot(P, V, axes=(2, 0)) sums P[s,a,s'] * V[s'] over s'
            # -> result shape (num_states, num_actions), exactly Q's shape
            Q = env.R + self.gamma * np.tensordot(env.P, V, axes=(2, 0))
            V_new = np.max(Q, axis=1)
 
            delta = np.max(np.abs(V_new - V))
            history.append(delta)
            V = V_new
 
            if delta < self.theta:
                print(f"Value Iteration converged in {i + 1} iterations!")
                break
 
        final_Q = env.R + self.gamma * np.tensordot(env.P, V, axes=(2, 0))
        optimal_policy = np.argmax(final_Q, axis=1)
        return V, optimal_policy, history
 
    def policy_iteration(self):
        """
        Alternates:
          1. Policy Evaluation: solve (I - gamma*P_pi) V = R_pi exactly with scipy
          2. Policy Improvement: pick the best action per state given new V
        Repeat until the policy stops changing.
        """
        env = self.env
        policy = np.zeros(env.num_states, dtype=int)  # start: everyone points "Up"
        V = np.zeros(env.num_states)
        state_indices = np.arange(env.num_states)
 
        for i in range(self.max_iters):
            # Slice out only the transitions/rewards for the action each state
            # currently uses under this policy
            P_pi = env.P[state_indices, policy, :]   # shape (num_states, num_states)
            R_pi = env.R[state_indices, policy]       # shape (num_states,)
 
            # Solve the linear system A V = R_pi  where A = I - gamma * P_pi
            A = np.eye(env.num_states) - self.gamma * P_pi
            V = scipy.linalg.solve(A, R_pi)
 
            # Policy improvement: re-check every state, keep the best action
            Q = env.R + self.gamma * np.tensordot(env.P, V, axes=(2, 0))
            new_policy = np.argmax(Q, axis=1)
 
            if np.array_equal(new_policy, policy):
                print(f"Policy Iteration converged in {i + 1} steps!")
                break
            policy = new_policy
 
        return V, policy
 
 
if __name__ == "__main__":
    # Initialize the stochastic Grid-World
    env = VectorizedStochasticGridWorld(height=10, width=10, slip_prob=0.15, step_cost=-0.1)
 
    # Analyze multiple discount factors gamma
    discount_factors = [0.9, 0.95, 0.99, 0.999]
 
    for gamma in discount_factors:
        print(f"\n--- Running Dynamic Programming for Gamma = {gamma} ---")
        solver = DynamicProgrammingSolver(env, gamma=gamma)
 
        # Run Value Iteration
        v_vi, pi_vi, hist_vi = solver.value_iteration()
        # Run Policy Iteration
        v_pi, pi_pi = solver.policy_iteration()
 
        # Verify that Value Iteration and Policy Iteration yield mathematically identical values
        max_diff = np.max(np.abs(v_vi - v_pi))
        print(f"Max Value discrepancy between VI and PI: {max_diff:.2e}")
 
    # Render the optimal path topology
    solver = DynamicProgrammingSolver(env, gamma=0.99)
    optimal_values, optimal_policy = solver.policy_iteration()
 
    # Plot final State Values Heatmap
    plt.figure(figsize=(8, 6))
    plt.imshow(optimal_values.reshape((10, 10)), cmap="viridis", origin="upper")
    plt.colorbar(label="Expected Value $V(s)$")
    plt.title(r"Vectorized Dynamic Programming: Optimal Value Grid ($\gamma=0.99$)")
 
    # Render Policy arrows
    arrows = {0: "\u2191", 1: "\u2193", 2: "\u2190", 3: "\u2192"}
    for r in range(10):
        for c in range(10):
            idx = r * 10 + c
            if idx == env.goal_state:
                label = "G"
            elif idx == env.trap_state:
                label = "T"
            else:
                label = arrows[optimal_policy[idx]]
            plt.text(c, r, label, ha="center", va="center",
                     color="white" if optimal_values[idx] < 0 else "black",
                     fontweight="bold")
 
    print("\nState Values visual plot saved as 'mdp_optimal_values.png'.")
    plt.savefig("mdp_optimal_values.png")
    plt.close()


--- Running Dynamic Programming for Gamma = 0.9 ---
Value Iteration converged in 52 iterations!
Max Value discrepancy between VI and PI: 8.68e-09

--- Running Dynamic Programming for Gamma = 0.95 ---
Value Iteration converged in 57 iterations!
Max Value discrepancy between VI and PI: 8.67e-09

--- Running Dynamic Programming for Gamma = 0.99 ---
Value Iteration converged in 61 iterations!
Max Value discrepancy between VI and PI: 1.19e-08

--- Running Dynamic Programming for Gamma = 0.999 ---
Value Iteration converged in 62 iterations!
Max Value discrepancy between VI and PI: 1.20e-08

State Values visual plot saved as 'mdp_optimal_values.png'.
